In [ ]:
import polars as pl
import os
import numpy as np

BASE_DIR = "/kaggle/input/home-credit-credit-risk-model-stability/parquet_files/train"
AGGR_DIR = "/kaggle/working/train/aggregated"
os.makedirs(AGGR_DIR, exist_ok=True)

def set_table_dtypes(lf: pl.LazyFrame) -> pl.LazyFrame:
    lf_schema = lf.collect_schema()
    lf_names = lf_schema.names()
    lf_dtypes = lf_schema.dtypes()
    
    cast_exprs = []

    for col, dtype in lf_schema.items():
        if col in ["case_id", "WEEK_NUM", "num_group1", "num_group2"]:
            cast_exprs.append(pl.col(col).cast(pl.Int64))
        elif col in ["date_decision"]:
            cast_exprs.append(pl.col(col).cast(pl.Date))
        elif col[-1] in ("P", "A"):
            cast_exprs.append(pl.col(col).cast(pl.Float64))
        elif col[-1] in ("M",):
            cast_exprs.append(pl.col(col).cast(pl.String))
        elif col[-1] in ("D",):
            cast_exprs.append(pl.col(col).cast(pl.Date))
        elif col[-1] in ("T", "L") and dtype == pl.Null:
            cast_exprs.append(pl.col(col).cast(pl.Float64))
    return lf.with_columns(cast_exprs)


def build_agg_exprs(tbl: pl.LazyFrame) -> list:
    """
    Build aggregation expressions for depth>0 table
    """
    cols = tbl.collect_schema().names()

    cols_A = [c for c in cols if c.endswith("A")]
    cols_P = [c for c in cols if c.endswith("P")]
    cols_D = [c for c in cols if c.endswith("D")]
    cols_M = [c for c in cols if c.endswith("M")]
    cols_TL = [c for c in cols if c.endswith(("T", "L"))]

    agg_exprs = []
    # Amount-like
    agg_exprs += [
        pl.col(c).sum().alias(f"{c}_sum") for c in cols_A
    ] + [
        pl.col(c).mean().alias(f"{c}_mean") for c in cols_A
    ] + [
        pl.col(c).max().alias(f"{c}_max") for c in cols_A
    ]

    # DPD-like
    agg_exprs += [
        pl.col(c).max().alias(f"{c}_max") for c in cols_P
    ] + [
        pl.col(c).mean().alias(f"{c}_mean") for c in cols_P
    ]

    # Date-like (assuming numeric or Date)
    agg_exprs += [
        pl.col(c).min().alias(f"{c}_min") for c in cols_D
    ] + [
        pl.col(c).max().alias(f"{c}_max") for c in cols_D
    ]

    # Masked categories
    agg_exprs += [  
        pl.col(c).n_unique().alias(f"{c}_nunique") for c in cols_M

    ] + [
    #Should be unique but this is memory heavy op, for now max
        pl.col(c).max().alias(f"{c}_max") for c in cols_M
    ]

    # Other numeric transforms
    agg_exprs += [
        pl.col(c).mean().alias(f"{c}_mean") for c in cols_TL
    ] + [
        pl.col(c).max().alias(f"{c}_max") for c in cols_TL
    ]

    return agg_exprs
    
def filter_cols(lf: pl.LazyFrame) -> pl.LazyFrame:
    protected = {"target", "case_id", "WEEK_NUM", "date_decision"}

    schema = lf.collect_schema()

    #add null_cols to drop list
    drop_cols = [
        name
        for name, dtype in zip(schema.names(), schema.dtypes())
        if dtype == pl.Null and name not in protected
    ]

    # get string columns from schema
    str_cols = [
        name
        for name, dtype in zip(schema.names(), schema.dtypes())
        if dtype == pl.Utf8 and name not in protected
    ]

    # compute n_unique for string columns
    nunique_df = lf.select(
        [pl.col(c).n_unique().alias(c) for c in str_cols]
    ).collect()

    # drop string cols that have more than 200 unqiue values (phones, IDs etc.)
    for c in str_cols:
        freq = nunique_df[0, c]
        if freq == 1 or freq > 200:
            drop_cols.append(c)

    # return lazy frame with those columns dropped
    print(f"Dropped {len(drop_cols)} columns")

    return lf.drop(drop_cols)

def reduce_mem_usage(lf: pl.LazyFrame) -> pl.LazyFrame:
    schema = lf.collect_schema()
    names = schema.names()
    dtypes = schema.dtypes()

    int_cols = [
        c for c, dt in zip(names, dtypes)
        if dt in (pl.Int64, pl.Int32, pl.Int16, pl.Int8)
    ]
    float_cols = [
        c for c, dt in zip(names, dtypes)
        if dt in (pl.Float64, pl.Float32)
    ]

    # nothing to do
    if not int_cols and not float_cols:
        return lf

    # one pass to get min/max for all numeric columns
    stats_exprs = []
    for c in int_cols + float_cols:
        stats_exprs.append(pl.col(c).min().alias(f"{c}__min"))
        stats_exprs.append(pl.col(c).max().alias(f"{c}__max"))

    stats = lf.select(stats_exprs).collect()

    cast_exprs = []

    # downcast integers
    for c in int_cols:
        c_min = stats[0, f"{c}__min"]
        c_max = stats[0, f"{c}__max"]
        if c_min is None or c_max is None:
            continue

        if np.iinfo(np.int8).min <= c_min <= np.iinfo(np.int8).max and \
           np.iinfo(np.int8).min <= c_max <= np.iinfo(np.int8).max:
            cast_exprs.append(pl.col(c).cast(pl.Int8))
        elif np.iinfo(np.int16).min <= c_min <= np.iinfo(np.int16).max and \
             np.iinfo(np.int16).min <= c_max <= np.iinfo(np.int16).max:
            cast_exprs.append(pl.col(c).cast(pl.Int16))
        elif np.iinfo(np.int32).min <= c_min <= np.iinfo(np.int32).max and \
             np.iinfo(np.int32).min <= c_max <= np.iinfo(np.int32).max:
            cast_exprs.append(pl.col(c).cast(pl.Int32))

    # downcast floats: Polars doesn't have float16, so 64 -> 32 where possible
    for c in float_cols:
        dt = schema[c]
        if dt != pl.Float64:
            continue  # already 32-bit or smaller

        c_min = stats[0, f"{c}__min"]
        c_max = stats[0, f"{c}__max"]
        if c_min is None or c_max is None:
            continue

        # if representable as float32, cast down
        if np.isfinite(c_min) and np.isfinite(c_max):
            if np.finfo(np.float32).min <= c_min <= np.finfo(np.float32).max and \
               np.finfo(np.float32).min <= c_max <= np.finfo(np.float32).max:
                cast_exprs.append(pl.col(c).cast(pl.Float32))

    if not cast_exprs:
        return lf

    return lf.with_columns(cast_exprs)
    
def handle_dates(lf: pl.LazyFrame) -> pl.LazyFrame:
    schema = lf.collect_schema()
    cols = schema.names()

    # If there is no date_decision in this table, nothing to do
    if "date_decision" not in cols:
        # still safely drop MONTH if present
        to_drop = [c for c in ("date_decision", "MONTH") if c in cols]
        return lf.drop(to_drop) if to_drop else lf

    # All columns ending with 'D' except the base decision date itself
    d_cols = [c for c in cols if c.endswith("D") and c != "date_decision"]

    #Convert all dates to days since decision date
    if d_cols:
        # colD := (colD - date_decision).dt.total_days()
        lf = lf.with_columns([
            (pl.col(c) - pl.col("date_decision")).dt.total_days().alias(c)
            for c in d_cols
        ])

    # Drop helper columns we don't want as features
    to_drop = [c for c in ("date_decision", "MONTH") if c in lf.collect_schema().names()]
    if to_drop:
        lf = lf.drop(to_drop)

    return lf

def process_group(group: list, depth=0, prefix: str="train"):
    """
    Scan parquets of same logical table, aggregate by case_id, concat, pre-process, write to disk
    Returns aggregated file path
    """
    
    #read raw files
    raw_paths = [os.path.join(BASE_DIR, f"{prefix}_{p}") for p in group]
    
    batches = []
    for p in raw_paths:
        lf = pl.scan_parquet(p)
        lf = set_table_dtypes(lf)
        if depth != 0:
            agg_exprs = build_agg_exprs(lf)
            lf = (
              lf
              .group_by("case_id")
              .agg(agg_exprs)
            )
        batches.append(lf)

    if len(batches) > 1:
        tbl = pl.concat(batches, how="vertical_relaxed")
        tbl = tbl.unique(subset=["case_id"])
    else:
        tbl = batches[0]

    #materialize here and do further steps for normal pd
    #filter some columns
    tbl = filter_cols(tbl)
    #process dates
    tbl = handle_dates(tbl)
    #reduce memory
    tbl = reduce_mem_usage(tbl)

    
    #write aggr file
    filename = os.path.splitext(group[0])[0] #e.g. applprev_1_0
    agg_path = os.path.join(AGGR_DIR, f"aggr_{filename}.parquet")
    tbl.sink_parquet(agg_path, mkdir=True)
    print(f"Aggregated {group} -> {agg_path}")

    return agg_path

In [ ]:
#
#
# Aggregate every table, remove stupid columns, 
#
#

BASE_F = "base.parquet"
DEPTH0_F = [
    ["static_0_0.parquet", "static_0_1.parquet"],
    ["static_cb_0.parquet"]
]
DEPTH1_F = [
    [
        "applprev_1_0.parquet",
        "applprev_1_1.parquet",
    ],
    ["other_1.parquet"],
    ["deposit_1.parquet"],
    ["person_1.parquet"],
    ["debitcard_1.parquet"],
    ["tax_registry_a_1.parquet"],
    ["tax_registry_b_1.parquet"],
    ["tax_registry_c_1.parquet"],
    [
        "credit_bureau_a_1_0.parquet",
        "credit_bureau_a_1_1.parquet",
        "credit_bureau_a_1_2.parquet",
        "credit_bureau_a_1_3.parquet",
    ],
    [
        "credit_bureau_b_1.parquet"
    ],
    ["credit_bureau_b_2.parquet"],
    [
        "credit_bureau_a_2_0.parquet",
        "credit_bureau_a_2_1.parquet",
        "credit_bureau_a_2_2.parquet",
        "credit_bureau_a_2_3.parquet",
        "credit_bureau_a_2_4.parquet",
        "credit_bureau_a_2_5.parquet",
        "credit_bureau_a_2_6.parquet",
        "credit_bureau_a_2_7.parquet",
        "credit_bureau_a_2_8.parquet",
        "credit_bureau_a_2_9.parquet",
        "credit_bureau_a_2_10.parquet"
    ],
    ["applprev_2.parquet"],
    ["person_2.parquet"]
]


#clean aggregated dir
!rm -r /kaggle/working/train/aggregated/*

agg_paths = []
for group in DEPTH0_F:
    agg_path = process_group(group, depth=0,prefix="train")
    agg_paths.append(agg_path)

for group in DEPTH1_F:
    agg_path = process_group(group, depth=1, prefix="train")
    agg_paths.append(agg_path)

#Build base
lf = pl.scan_parquet(os.path.join(BASE_DIR, f"train_{BASE_F}"))

print("Joining agg files")
agg_lfs = [pl.scan_parquet(p) for p in agg_paths]
for agg_lf in agg_lfs:
    lf = lf.join(agg_lf, on="case_id", how="left")

In [ ]:
#remember schema to align test dataset
TRAIN_SCHEMA = lf.collect_schema()

#Materialize batch
df = lf.slice(0, 800_000).collect() #limit number of rows, idea is that more feats > many rows

In [1]:
#
#
# Split test just to see AUC before submitting
#
#

import numpy as np
from glob import glob

def df_to_xy(df: pl.DataFrame):
    #casting dates if any left to days since epoch
    df_num = df.with_columns([
        pl.col(pl.Date, pl.Datetime).cast(pl.Int64),
        pl.col(pl.Boolean).cast(pl.Int8),
    ])

    
    # encode string to int
    str_cols = [
        c for c, dt in zip(df_num.columns, df_num.dtypes)
        if dt == pl.Utf8
    ]
    if str_cols:
        df_num = df_num.with_columns([
            pl.col(c)
            .cast(pl.Categorical)
            .to_physical()
            .cast(pl.Int32) 
            .alias(c)
            for c in str_cols
        ])
        
    #for use in both test and train df
    if "target" in df.columns:
        to_exclude = ["target", "case_id", "WEEK_NUM"]
        y = df_num["target"].to_numpy()
    else:
        to_exclude = ["case_id", "WEEK_NUM"]
        y = None

    #exclude
    X = df_num.select(
        pl.all()
        .exclude(to_exclude)
        #.exclude(pl.Utf8)
    ).to_numpy()
    
    return X, y
    
#Shall we shuffle?
#df_shuffled = df.sample(frac=1.0, shuffle=True, seed=42)
df_train = df.slice(0, 780_000)
df_val = df.slice(780_000, 790_000)
df_test = df.slice(790_000, 800_000)

X_train, y_train = df_to_xy(df_train)
X_val, y_val = df_to_xy(df_val)
X_test, y_test = df_to_xy(df_test)

NameError: name 'pl' is not defined

In [ ]:
#
#
# Train 2 lGBM + 2 Cat with soft voting on limited slice
#
#

import numpy as np
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score

# -----------------------------
# 1) Common imbalance handling
# -----------------------------
n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
scale_pos_weight = n_neg / n_pos

# -----------------------------
# 2) Two LightGBM models (lgb.train)
# -----------------------------
base_lgb_params = {
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.05,
    "scale_pos_weight": scale_pos_weight,
    "num_leaves": 64,
    "min_data_in_leaf": 500,
    "feature_fraction": 0.7,
    "bagging_fraction": 0.7,
    "bagging_freq": 1,
    "lambda_l1": 1.0,
    "lambda_l2": 1.0,
    "verbosity": -1,
}

params_lgb1 = {
    **base_lgb_params,
    "seed": 42,
}
params_lgb2 = {
    **base_lgb_params,
    "seed": 2024,
    "feature_fraction": 0.6,   # small tweaks to add diversity
    "bagging_fraction": 0.8,
}

dtrain = lgb.Dataset(X_train, label=y_train, free_raw_data=False)
dvalid = lgb.Dataset(X_val,   label=y_val,   free_raw_data=False)

callbacks = [lgb.early_stopping(stopping_rounds=100, verbose=False)]

model_lgb1 = lgb.train(
    params_lgb1,
    dtrain,
    num_boost_round=1000,
    valid_sets=[dvalid],
    valid_names=["valid"],
    callbacks=callbacks,
    keep_training_booster=False,
)

model_lgb2 = lgb.train(
    params_lgb2,
    dtrain,
    num_boost_round=1000,
    valid_sets=[dvalid],
    valid_names=["valid"],
    callbacks=callbacks,
    keep_training_booster=False,
)

print("LGB1 best AUC:", model_lgb1.best_score["valid"]["auc"])
print("LGB2 best AUC:", model_lgb2.best_score["valid"]["auc"])

# -----------------------------
# 3) Two CatBoost models
# -----------------------------
cat_base_params = {
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "learning_rate": 0.05,
    "iterations": 1000,
    "scale_pos_weight": scale_pos_weight,
    "random_seed": 42,
    "verbose": 100,
    "depth": 8,
    "l2_leaf_reg": 3.0,
    "task_type": "CPU",
}

cat1 = CatBoostClassifier(**cat_base_params)
cat2 = CatBoostClassifier(**{**cat_base_params, "random_seed": 2024, "depth": 6})

cat1.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    early_stopping_rounds=100,
    verbose=100,
)
cat2.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    early_stopping_rounds=100,
    verbose=100,
)

# -----------------------------
# 4) Soft voting on validation
# -----------------------------
pred_val_lgb1 = model_lgb1.predict(X_val)
pred_val_lgb2 = model_lgb2.predict(X_val)
pred_val_cat1 = cat1.predict_proba(X_val)[:, 1]
pred_val_cat2 = cat2.predict_proba(X_val)[:, 1]

pred_val_ens = (pred_val_lgb1 + pred_val_lgb2 + pred_val_cat1 + pred_val_cat2) / 4.0
val_auc = roc_auc_score(y_val, pred_val_ens)
print("Ensemble AUC (val):", val_auc)

In [ ]:
#
#
# See preliminary results
#
#

from sklearn.metrics import roc_auc_score, log_loss, confusion_matrix

pred_test_lgb1 = model_lgb1.predict(X_test)
pred_test_lgb2 = model_lgb2.predict(X_test)
pred_test_cat1 = cat1.predict_proba(X_test)[:, 1]
pred_test_cat2 = cat2.predict_proba(X_test)[:, 1]

y_pred_proba = (pred_test_lgb1 + pred_test_lgb2 + pred_test_cat1 + pred_test_cat2) / 4.0

print("Test logloss:", log_loss(y_test, y_pred_proba))
print("Test AUC:", roc_auc_score(y_test, y_pred_proba))

threshold = 0.7
y_pred = (y_pred_proba >= threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

print(f"TP: {tp}")
print(f"FP: {fp}")
print(f"TN: {tn}")
print(f"FN: {fn}")

In [ ]:
#
#
# Preparing test dataset
#
#

def align_to_train_schema(df: pl.DataFrame, schema: pl.Schema) -> pl.DataFrame:
    ref_names = schema.names()
    ref_dtypes = schema.dtypes()

    exprs = []
    for name, dtype in zip(ref_names, ref_dtypes):
        if name == "target":
            # no target in real test batches
            continue

        if name in df.columns:
            # Cast to train dtype (handles pl.Null -> proper dtype as well)
            exprs.append(pl.col(name).cast(dtype).alias(name))
        else:
            # Column missing in test → create typed-null column
            exprs.append(pl.lit(None, dtype=dtype).alias(name))

    # Only keep columns that exist in the ref schema (minus target),
    # and in the same order.
    aligned = df.select(exprs)
    return aligned

BASE_DIR = "/kaggle/input/home-credit-credit-risk-model-stability/parquet_files/test"
AGGR_DIR = "/kaggle/working/test/aggregated"
os.makedirs(AGGR_DIR, exist_ok=True)

!rm -r /kaggle/working/test/aggregated/*

#build aggreagated tables
agg_paths_test = []

for group in DEPTH0_F:
    agg_path = process_group(group, depth=0,prefix="test")
    agg_paths_test.append(agg_path)
for group in DEPTH1_F:
    agg_path = process_group(group, depth=1, prefix="test")
    agg_paths_test.append(agg_path)

#Build base
lf = pl.scan_parquet(os.path.join(BASE_DIR, f"test_{BASE_F}"))

#join to base
print("Joining agg files")
agg_lfs = [pl.scan_parquet(p) for p in agg_paths_test]
for agg_lf in agg_lfs:
    lf = lf.join(agg_lf, on="case_id", how="left")

df_submit = lf.collect()

df_submit = align_to_train_schema(df_submit, TRAIN_SCHEMA)

case_ids_submit = df_submit["case_id"].to_numpy()
X_submit, _ = df_to_xy(df_submit)

# -----------------------------
# 5) Soft voting for submission
# -----------------------------
pred_submit_lgb1 = model_lgb1.predict(X_submit)
pred_submit_lgb2 = model_lgb2.predict(X_submit)
pred_submit_cat1 = cat1.predict_proba(X_submit)[:, 1]
pred_submit_cat2 = cat2.predict_proba(X_submit)[:, 1]

y_submit = (pred_submit_lgb1 + pred_submit_lgb2 +
                pred_submit_cat1 + pred_submit_cat2) / 4.0

print(y_submit)

import pandas as pd
submission = pd.DataFrame({
    "case_id": case_ids_submit,
    "score": y_submit,
})

submission.to_csv("/kaggle/working/submission.csv", index=False)